<a href="https://colab.research.google.com/github/JManuelRG/Procesos-de-Separacion/blob/main/Procesos%20de%20Absorci%C3%B3n%20y%20Desorci%C3%B3n/Absorci%C3%B3n-Desorci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Absorción y desorción de gases

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JManuelRG/Procesos-de-Separacion/blob/main/02_absorcion_desorcion/01_absorcion_desorcion.ipynb)

**Módulo 02 · Absorción-desorción**

## Objetivos

Al terminar este cuaderno podrás:

1. Plantear **balances de materia** en absorbedores y desorbedores usando razones molares.
2. Representar la **línea de operación** junto a la **línea de equilibrio** e interpretar su posición relativa.
3. Calcular el **flujo mínimo de líquido** (absorción) y el **flujo máximo** (desorción), y elegir un flujo de operación.
4. Determinar el **número de etapas ideales** con el método escalonado de McCabe-Thiele y con la ecuación de Kremser.
5. Convertir etapas ideales en platos reales con la **eficiencia global** y la **eficiencia de Murphree**.
6. Estimar el efecto del **calor de disolución** mediante un balance de energía.

## Contenido

| Parte | Descripción |
|---|---|
| **1** | Teoría detallada |
| **2** | Ejemplos resueltos paso a paso (siete ejemplos, varios conectados entre sí) |
| **3** | Ejercicios propuestos con respuestas |
| **4** | Resumen y referencias |

## Preparación del entorno

Se usan `numpy`, `scipy` y `matplotlib`, que ya vienen instalados en Google Colab. La celda siguiente define también las **funciones de apoyo** que se reutilizan en todo el cuaderno.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

R = 8.314  # J/(mol·K)

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

# ------------------------------------------------------------------
# Equilibrio (Ley de Henry en fracciones molares, y = m·x)
# expresado en razones molares: Y* = m·X / (1 + (1 - m)·X)
# ------------------------------------------------------------------
def Y_eq(X, m):
    return m * X / (1 + (1 - m) * X)

def X_eq(Y, m):
    return Y / (m + Y * (m - 1))

# ------------------------------------------------------------------
# Número de etapas ideales por el método escalonado (McCabe-Thiele).
# Funciona para absorción (Yb > Yt) y para desorción (Yb < Yt).
#   Gs, Ls : flujos de gas inerte y de disolvente (base libre de soluto)
#   Yb, Yt : razón molar del gas en el fondo y en el tope
#   Xt     : razón molar del líquido que entra por el tope
#   eq     : función de equilibrio  Y* = eq(X)
# Las etapas se cuentan desde el fondo.
# ------------------------------------------------------------------
def etapas_ideales(Gs, Ls, Yb, Yt, Xt, eq, nmax=200):
    pend = Ls / Gs
    Xb = Xt + Gs * (Yb - Yt) / Ls          # balance de soluto
    X = Xb
    X_et, Y_et, Xprev_et = [], [], []
    absorcion = Yb > Yt
    for n in range(1, nmax + 1):
        Y = eq(X)                           # gas que sale de la etapa n (en equilibrio con X)
        Xprev = Xt + (Y - Yt) / pend        # línea de operación
        X_et.append(X); Y_et.append(Y); Xprev_et.append(Xprev)
        terminado = (Xprev <= Xt) if absorcion else (Xprev >= Xt)
        if terminado:
            frac = (X - Xt) / (X - Xprev)   # fracción de la última etapa
            return dict(N=(n - 1) + frac, Xb=Xb, X=X_et, Y=Y_et, Xprev=Xprev_et,
                        pend=pend, Yb=Yb, Yt=Yt, Xt=Xt)
        X = Xprev
    return dict(N=np.inf, Xb=Xb, X=X_et, Y=Y_et, Xprev=Xprev_et, pend=pend, Yb=Yb, Yt=Yt, Xt=Xt)

def trazo_escalones(res):
    # Coordenadas de la escalera para graficarla
    xs, ys = [res["Xb"]], [res["Yb"]]
    absorcion = res["Yb"] > res["Yt"]
    for X, Y, Xp in zip(res["X"], res["Y"], res["Xprev"]):
        xs.append(X); ys.append(Y)                       # tramo vertical hasta el equilibrio
        Xp = max(Xp, res["Xt"]) if absorcion else min(Xp, res["Xt"])
        xs.append(Xp); ys.append(Y)                      # tramo horizontal hasta la línea de operación
    return xs, ys

# ------------------------------------------------------------------
# Ecuación de Kremser (equilibrio y operación lineales)
#   F = A = L/(m·G)  en absorción     |   F = S = m·G/L  en desorción
#   phi = fracción del soluto transferido respecto al máximo posible
# ------------------------------------------------------------------
def kremser_N(phi, F):
    if abs(F - 1) < 1e-9:
        return phi / (1 - phi)
    return np.log((1 - phi / F) / (1 - phi)) / np.log(F)

def kremser_phi(N, F):
    if abs(F - 1) < 1e-9:
        return N / (N + 1)
    return (F ** (N + 1) - F) / (F ** (N + 1) - 1)

print("Funciones de apoyo cargadas.")

---
# PARTE 1 · TEORÍA

## 1. Introducción

### 1.1 ¿Qué son la absorción y la desorción?

- **Absorción:** un componente (soluto) pasa de una mezcla gaseosa a un líquido (disolvente) en el que es soluble. Se usa para **eliminar o recuperar** un componente del gas.
- **Desorción** (también *stripping* o agotamiento): el proceso inverso. Un soluto disuelto en un líquido pasa a una corriente gaseosa (aire, vapor, nitrógeno).

La fuerza impulsora es la **diferencia entre la composición real y la de equilibrio** entre las fases, y el soluto se transfiere en el sentido que la reduce.

### 1.2 Aplicaciones

| Operación | Ejemplos |
|---|---|
| Absorción | Lavado de gases ácidos con soluciones de aminas, eliminación de NH₃ con agua, recuperación de disolventes, absorción de oxígeno en un biorreactor |
| Desorción | Eliminación de compuestos orgánicos volátiles del agua, desgasificación de líquidos, regeneración del disolvente después de absorber |

Con frecuencia las dos operaciones forman un **ciclo**: el disolvente absorbe en una columna, se regenera en una segunda columna por desorción y regresa a la primera.

### 1.3 Equipos de contacto

| Tipo | Contacto | Se diseña por |
|---|---|---|
| **Columnas de platos** | por **etapas** discretas | número de etapas ideales y eficiencia |
| **Columnas empacadas** | **continuo** | altura de unidad de transferencia (HTU) y NTU |

Este cuaderno desarrolla el **método por etapas**. Su resultado, el número de etapas ideales, también es el punto de partida para estimar la altura de una columna empacada mediante la altura equivalente a un plato teórico (HETP). El diseño continuo con HTU/NTU se verá en el módulo de columnas de absorción.

### 1.4 Esquema de una columna en contracorriente

```
          Líquido pobre (L_s, X_t)              Gas limpio (G_s, Y_t)
                    │                                   ▲
                    ▼                                   │
              ┌───────────┐ ◄── TOPE (t)
              │  etapa 1  │
              │  etapa 2  │
              │    ...    │
              │  etapa N  │
              └───────────┘ ◄── FONDO (b)
                    │                                   ▲
                    ▼                                   │
          Líquido rico (L_s, X_b)               Gas rico (G_s, Y_b)
```

El gas **sube** y el líquido **baja**: es el arreglo en **contracorriente**.

## 2. Notación y bases de cálculo

### 2.1 Razones molares

Las fracciones molares $y$ y $x$ no son proporcionales a la cantidad de soluto cuando este es abundante, porque el flujo total cambia a lo largo de la columna. Por eso se trabaja con **razones molares**, que se refieren a una base **libre de soluto**:

$$
Y = \frac{y}{1-y} = \frac{\text{mol de soluto}}{\text{mol de gas inerte}},
\qquad
X = \frac{x}{1-x} = \frac{\text{mol de soluto}}{\text{mol de disolvente}}
$$

Las conversiones inversas son $y = \dfrac{Y}{1+Y}$ y $x = \dfrac{X}{1+X}$.

### 2.2 Flujos

| Símbolo | Significado | ¿Cambia a lo largo de la columna? |
|---|---|---|
| $G$, $L$ | flujos molares **totales** de gas y líquido | Sí |
| $G_s = G(1-y)$ | flujo de gas **inerte** (sin soluto) | **No** |
| $L_s = L(1-x)$ | flujo de **disolvente** (sin soluto) | **No** |

Como $G_s$ y $L_s$ son constantes, los balances de materia quedan **lineales** en $X$ y $Y$.

### 2.3 Subíndices

- **b**: fondo de la columna (*bottom*).
- **t**: tope de la columna (*top*).

En un **absorbedor**, el gas rico entra por el fondo ($Y_b$) y sale limpio por el tope ($Y_t$); el líquido pobre entra por el tope ($X_t$) y sale rico por el fondo ($X_b$).

## 3. Equilibrio gas-líquido

### 3.1 Ley de Henry

Para soluciones diluidas, la presión parcial de A en el gas es proporcional a su fracción molar en el líquido:

$$
p_A = H\,x_A
\;\;\Longrightarrow\;\;
y_A = \frac{H}{P}\,x_A = m\,x_A
$$

donde $H$ es la constante de Henry (depende de la temperatura) y $m = H/P$ es la **pendiente de equilibrio**, adimensional.

### 3.2 Equilibrio en razones molares

Al sustituir $y = Y/(1+Y)$ y $x = X/(1+X)$ en $y = m\,x$ y despejar $Y$:

$$
\boxed{\,Y^{*} = \frac{m\,X}{1+(1-m)\,X}\,}
\qquad\text{y}\qquad
X^{*} = \frac{Y}{m+(m-1)\,Y}
$$

Para soluciones **muy diluidas** ($X,Y\ll 1$) se reduce a $Y^* \approx m\,X$. En el diagrama $Y$-$X$ la **curva de equilibrio** es una curva suave que, si $m>1$, se curva hacia arriba.

### 3.3 Efecto de la temperatura y la presión

| Variable | Efecto sobre $m$ | Consecuencia |
|---|---|---|
| Aumentar $T$ | $m$ **aumenta** (el gas es menos soluble) | desfavorece la absorción y favorece la desorción |
| Aumentar $P$ | $m=H/P$ **disminuye** | favorece la absorción |

Por eso la absorción se hace a **alta presión y baja temperatura**, y la desorción a **baja presión y alta temperatura** (por ejemplo, con vapor).

### 3.4 Soluto con mucha solubilidad o poca

| $m$ | Solubilidad del gas | Controla la resistencia |
|---|---|---|
| pequeña | alta | fase gas |
| grande | baja | fase líquida |

Esto conecta con la teoría de las dos películas del cuaderno anterior.

## 4. Balance de materia y línea de operación

### 4.1 Balance global de soluto

Entra soluto con el gas y con el líquido, y sale de la misma forma. Como $G_s$ y $L_s$ son constantes:

$$
\boxed{\,G_s\,(Y_b - Y_t) = L_s\,(X_b - X_t)\,}
$$

Con esta ecuación se calcula la composición de la corriente que no se especifica.

### 4.2 Recuperación

La fracción del soluto del gas que se transfiere al líquido es:

$$
\varphi = \frac{Y_b - Y_t}{Y_b}
$$

### 4.3 Línea de operación

Un balance de soluto entre el **tope** de la columna y una sección cualquiera (donde las composiciones son $X$ y $Y$) da:

$$
\boxed{\,Y = Y_t + \frac{L_s}{G_s}\,(X - X_t)\,}
$$

Es una **recta** con pendiente $L_s/G_s$ que pasa por los puntos extremos $(X_t,\,Y_t)$ y $(X_b,\,Y_b)$. Para soluciones diluidas, $L\approx L_s$ y $G\approx G_s$, y se escribe en fracciones molares: $y = y_t + \dfrac{L}{G}(x - x_t)$.

### 4.4 Posición respecto al equilibrio

La línea de operación muestra las composiciones **reales** en cada sección. Su posición frente a la curva de equilibrio indica el sentido de la transferencia:

| Operación | Línea de operación | Razón |
|---|---|---|
| **Absorción** | **por encima** de la curva de equilibrio | $Y > Y^*$: el gas tiene más soluto que el de equilibrio y lo cede al líquido |
| **Desorción** | **por debajo** de la curva de equilibrio | $Y < Y^*$: el líquido cede soluto al gas |

La **distancia vertical** entre ambas curvas es la fuerza impulsora: mientras mayor es, menos etapas se necesitan.

## 5. Flujo mínimo de líquido (absorción) y flujo máximo (desorción)

### 5.1 Absorción: $(L_s/G_s)_{min}$

Si se **reduce** el flujo de líquido, la pendiente de la línea de operación disminuye y $X_b$ crece. El límite ocurre cuando la línea de operación **toca** la curva de equilibrio: el líquido sale en equilibrio con el gas de entrada. Allí la fuerza impulsora es nula y harían falta **infinitas etapas** (punto de pinzamiento).

$$
\boxed{\,\left(\frac{L_s}{G_s}\right)_{min} = \frac{Y_b - Y_t}{X_b^{*} - X_t}\,}
\qquad X_b^{*} = X_{eq}(Y_b)
$$

Si la curva de equilibrio tiene tramos que se curvan hacia abajo, el contacto puede ocurrir en un punto intermedio; entonces la pendiente mínima es la de la **tangente** a la curva trazada desde $(X_t,\,Y_t)$.

**Caso simple.** Con equilibrio lineal $Y^*=mX$ y $X_t = 0$:

$$
\left(\frac{L_s}{G_s}\right)_{min} = m\,\varphi
$$

### 5.2 Selección del flujo de operación

Un flujo mayor que el mínimo aumenta la fuerza impulsora y **reduce las etapas**, pero requiere más disolvente y más energía para regenerarlo. Se busca un compromiso económico:

$$
\frac{L_s}{G_s} = f\left(\frac{L_s}{G_s}\right)_{min}, \qquad f \approx 1.2 \text{ a } 2.0 \quad(\text{valor frecuente } f = 1.5)
$$

### 5.3 Desorción: $(L_s/G_s)_{max}$

En desorción la línea de operación está por debajo del equilibrio. Si se **aumenta** el flujo de líquido (o se reduce el de gas), la línea se acerca a la curva en el **tope**, donde el líquido que entra con $X_t$ queda en equilibrio con el gas que sale.

$$
\boxed{\,\left(\frac{L_s}{G_s}\right)_{max} = \frac{Y_t^{*} - Y_b}{X_t - X_b}\,}
\qquad Y_t^{*} = Y_{eq}(X_t)
$$

Equivale a un **flujo mínimo de gas**. En la práctica se usa $G_s \approx 1.2$ a $2$ veces el mínimo, es decir, $L_s/G_s$ por debajo del máximo.

## 6. Contacto en contracorriente y en corriente paralela

| Característica | Contracorriente | Corriente paralela (cocorriente) |
|---|---|---|
| Entrada de las corrientes | extremos opuestos | mismo extremo |
| Pendiente de la línea de operación | $+L_s/G_s$ | $-L_s/G_s$ (negativa) |
| Fuerza impulsora | mayor y más uniforme | grande a la entrada y casi nula a la salida |
| Transferencia máxima | puede acercarse al 100 % con muchas etapas | **limitada** por el equilibrio de salida |
| Uso | **estándar** en absorción y desorción | reacciones muy rápidas, o cuando basta una sola etapa |

**Límite de la corriente paralela.** Con equilibrio lineal $Y^*=mX$, $X_{in}=0$ y un número infinito de etapas, las corrientes salen en equilibrio y la fracción máxima absorbida es

$$
\varphi_{max} = \frac{L_s/G_s}{m + L_s/G_s}
$$

No importa cuánto se alargue la columna: **nunca** se supera ese valor.

## 7. Líneas de unión y fuerza impulsora en el contacto continuo

En un punto de la columna, la composición del gas es $Y$ y la del líquido $X$ (un punto sobre la línea de operación). La composición **en la interfase** $(X_i,\,Y_i)$ es un punto de la curva de equilibrio. Ambos se conectan con la **línea de unión**, cuya pendiente se obtiene de la teoría de las dos películas (cuaderno anterior):

$$
\text{pendiente} = -\frac{k_x}{k_y} \quad(\text{soluciones diluidas})
$$

Para soluciones concentradas, la pendiente es $-\dfrac{k_x'/(1-x)_{lm}}{k_y'/(1-y)_{lm}}$.

- Si $k_x\gg k_y$ (resistencia en el gas), la línea es casi **vertical** y $X_i\approx X$.
- Si $k_x\ll k_y$ (resistencia en el líquido), la línea es casi **horizontal** y $Y_i\approx Y$.

El segmento vertical entre la línea de operación y la curva de equilibrio es $Y-Y^*$, la fuerza impulsora global. Esta construcción se usa para calcular la altura de columnas empacadas.

## 8. Etapas ideales: método de McCabe-Thiele

### 8.1 Concepto de etapa ideal

Una **etapa ideal** (o teórica) es aquella en la que las corrientes que salen están en **equilibrio**: el gas que sale de la etapa $n$ ($Y_n$) está en equilibrio con el líquido que sale de ella ($X_n$):

$$
Y_n = Y^{*}(X_n)
$$

Las corrientes que se cruzan entre dos etapas consecutivas obedecen la línea de operación.

### 8.2 Construcción en el diagrama $Y$-$X$

1. Se traza la **curva de equilibrio**.
2. Se traza la **línea de operación** entre $(X_t,\,Y_t)$ y $(X_b,\,Y_b)$.
3. Se parte del extremo $(X_b,\,Y_b)$ y se sube o baja **verticalmente** hasta la curva de equilibrio: da el $Y_n$ del gas que sale de la última etapa.
4. Se avanza **horizontalmente** hasta la línea de operación: da el $X_{n-1}$ del líquido que entra a esa etapa.
5. Se repite hasta cruzar el otro extremo de la línea de operación. **Cada escalón es una etapa ideal**.
6. La fracción del último escalón se interpola.

### 8.3 Algoritmo numérico

El mismo procedimiento puede programarse. Partiendo de $X_N=X_b$ (con las etapas contadas desde el fondo):

$$
Y_n = Y^{*}(X_n), \qquad X_{n-1} = X_t + \frac{Y_n - Y_t}{L_s/G_s}
$$

y se detiene cuando $X_{n-1}$ alcanza $X_t$. Eso es lo que hace la función `etapas_ideales` definida al inicio.

### 8.4 Observaciones

- Si la línea de operación **cruza** o toca la curva de equilibrio, el procedimiento no converge: el flujo de líquido está por debajo del mínimo.
- Cuanto más cerca está la línea de operación de la curva, **más escalones** se necesitan.
- El método supone flujos molares constantes en razones molares, por eso funciona incluso con solutos concentrados.

## 9. Ecuación de Kremser (solución analítica)

Cuando **el equilibrio es lineal** ($Y^*=mX$) y la línea de operación es recta (soluciones diluidas), los balances etapa por etapa se resuelven de forma exacta.

### 9.1 Absorción

Se define el **factor de absorción** $A = \dfrac{L}{m\,G}$. La fracción absorbida, referida al máximo posible, es:

$$
\varphi = \frac{Y_b - Y_t}{Y_b - m\,X_t} = \frac{A^{N+1}-A}{A^{N+1}-1}
$$

Despejando $N$:

$$
\boxed{\,N = \frac{\ln\!\left[\dfrac{1-\varphi/A}{1-\varphi}\right]}{\ln A}\,}
\qquad(A\neq 1),
\qquad N = \frac{\varphi}{1-\varphi}\;\;(A=1)
$$

### 9.2 Desorción

Se define el **factor de desorción** $S = \dfrac{m\,G}{L}$, y la fracción desorbida es:

$$
\varphi = \frac{X_t - X_b}{X_t - Y_b/m} = \frac{S^{N+1}-S}{S^{N+1}-1},
\qquad
N = \frac{\ln\!\left[\dfrac{1-\varphi/S}{1-\varphi}\right]}{\ln S}
$$

### 9.3 Observaciones

| Factor | Interpretación |
|---|---|
| $A>1$ | hay líquido suficiente: se puede absorber todo el soluto con suficientes etapas |
| $A<1$ | el máximo absorbible con infinitas etapas es $\varphi=A$ |
| $A=1$ | línea de operación paralela al equilibrio |

El mismo razonamiento aplica a $S$ en desorción. En la práctica se diseña con $A\approx 1.2$ a $2$.

## 10. Eficiencia de etapas

Los platos reales no alcanzan el equilibrio. Por eso el número de platos reales es mayor que el de etapas ideales.

### 10.1 Eficiencia global

$$
E_o = \frac{N_{ideales}}{N_{reales}}
$$

Es un valor **promedio** para toda la columna, fácil de usar y de medir en una columna existente.

### 10.2 Eficiencia de Murphree (en fase gas)

Mide, **para un solo plato**, qué fracción del cambio ideal de composición se logra en el gas:

$$
\boxed{\,E_{MV} = \frac{Y_{n+1}-Y_n}{Y_{n+1}-Y_n^{*}}\,}, \qquad Y_n^{*} = Y^{*}(X_n)
$$

- $Y_{n+1}$: gas que **entra** al plato $n$.
- $Y_n$: gas que **sale** del plato.
- $Y_n^*$: composición de equilibrio con el líquido que sale del plato.

Si $E_{MV}=1$, el gas sale en equilibrio (etapa ideal). De manera análoga se define la eficiencia en fase líquida $E_{ML}$.

### 10.3 Relación entre ambas (Lewis)

Si el equilibrio y la línea de operación son rectas y $E_{MV}$ es constante en todos los platos:

$$
\boxed{\,E_o = \frac{\ln\left[1+E_{MV}\,(\lambda-1)\right]}{\ln\lambda}\,}, \qquad \lambda = \frac{m\,G}{L} = \frac{1}{A}
$$

### 10.4 Valores típicos

Las eficiencias dependen de la solubilidad del gas, de la viscosidad del líquido y del tipo de plato. Existen correlaciones empíricas para absorbedores y desorbedores, como la de O'Connell, y la práctica industrial es completarlas con datos de planta o de piloto.

## 11. Balance de energía y operación no isotérmica

### 11.1 ¿Por qué importa?

Al disolverse, el soluto libera **calor de disolución** ($\Delta H_{sol}<0$). Si la columna es adiabática, ese calor eleva la temperatura del líquido a medida que baja y **aumenta $m$**, es decir, reduce la solubilidad. El resultado: la curva de equilibrio se desplaza hacia arriba y la fuerza impulsora disminuye.

### 11.2 Balance global de energía

$$
G_b\,H_{G,b} + L_t\,H_{L,t} = G_t\,H_{G,t} + L_b\,H_{L,b}
$$

donde $H$ es la entalpía molar de cada corriente, que incluye el calor de disolución.

### 11.3 Estimación del aumento de temperatura

Si el calor sensible del gas y la evaporación del disolvente son pequeños, todo el calor liberado calienta el líquido:

$$
\Delta T \approx \frac{G_s\,(Y_b-Y_t)\,(-\Delta H_{sol})}{L_s\,M_{disolv}\,c_p}
$$

Es una **estimación conservadora**, pues la evaporación del disolvente y el calor sensible del gas reducen el aumento real.

### 11.4 Efecto sobre el equilibrio

Con la ecuación de van't Hoff, la pendiente de equilibrio varía con la temperatura:

$$
m(T) = m_0\,\exp\!\left[\frac{\Delta H_{sol}}{R}\left(\frac1T-\frac1{T_0}\right)\right]
$$

Con $\Delta H_{sol}<0$, $m$ crece al aumentar $T$.

### 11.5 Remedios

- Operar con **más líquido** (mayor $L_s/G_s$), que reparte el calor en más masa.
- **Enfriamiento intermedio** del líquido entre platos o sobre una recirculación.
- Entrar con el **líquido más frío**.
- En soluciones muy diluidas el aumento es pequeño y la operación puede tratarse como **isotérmica**.

## 12. Alimentaciones múltiples, salidas laterales y Ponchon-Savarit

### 12.1 Alimentaciones múltiples y salidas laterales

Si se agrega una corriente intermedia o se extrae una corriente lateral, los flujos $G_s$ o $L_s$ **cambian** a partir de ese punto. La línea de operación se divide en **tramos rectos con distinta pendiente**, que se unen en el nivel de la alimentación o de la salida. El conteo de etapas sigue igual, pero cada tramo usa su propia pendiente.

### 12.2 Método de Ponchon-Savarit

El método de McCabe-Thiele supone **flujos molares constantes** y operación isotérmica. Cuando los efectos térmicos son importantes (calores de disolución o vaporización grandes), esa hipótesis ya no es válida. El método de **Ponchon-Savarit** incorpora los balances de **entalpía** etapa por etapa y trabaja en un diagrama **entalpía-concentración**. Se desarrolla en un cuaderno aparte (`07_ponchon_savarit`).

## 13. Procedimiento de diseño

| Paso | Acción |
|---|---|
| 1 | Convertir composiciones a **razones molares** y calcular $G_s$. |
| 2 | Fijar la **recuperación** y calcular $Y_t$ (o $X_b$ en desorción). |
| 3 | Obtener la **curva de equilibrio** ($m$, $T$, $P$). |
| 4 | Calcular el flujo **mínimo** de líquido (o el máximo en desorción). |
| 5 | Elegir el flujo de operación con $f\approx1.2$–$2.0$. |
| 6 | Calcular $X_b$ por balance y trazar la **línea de operación**. |
| 7 | Determinar las **etapas ideales** (escalones o Kremser). |
| 8 | Convertir a **platos reales** con la eficiencia. |
| 9 | Revisar el **balance de energía** y el efecto de la temperatura. |
| 10 | Dimensionar la columna (diámetro, caída de presión, altura). |

## 14. Visualización de los conceptos

### 14.1 Absorción y desorción: posición de la línea de operación

Se usa un equilibrio con $m=2$. En absorción la línea de operación queda **sobre** la curva de equilibrio y en desorción **bajo** ella.

In [ ]:
m = 2.0
Xg = np.linspace(0, 0.05, 200)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

# --- Absorción ---
ax = axs[0]
Xt, Yt, Xb, Yb = 0.0, 0.0064, 0.0206, 0.0638
ax.plot(Xg, Y_eq(Xg, m), "k", lw=2, label="Equilibrio")
ax.plot([Xt, Xb], [Yt, Yb], "r-o", lw=2, label="Línea de operación")
ax.annotate("fondo", (Xb, Yb), xytext=(6, 4), textcoords="offset points")
ax.annotate("tope", (Xt, Yt), xytext=(6, 6), textcoords="offset points")
ax.fill_between([Xt, Xb], [Yt, Yb], [Y_eq(Xt, m), Y_eq(Xb, m)], color="tab:blue", alpha=0.12)
ax.set_title("ABSORCIÓN: operación sobre el equilibrio")
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.legend(loc="upper left")

# --- Desorción ---
ax = axs[1]
Xt2, Xb2, Yb2 = 0.03, 0.004, 0.0
pend = 1.2
Yt2 = Yb2 + pend * (Xt2 - Xb2)
ax.plot(Xg, Y_eq(Xg, m), "k", lw=2, label="Equilibrio")
ax.plot([Xb2, Xt2], [Yb2, Yt2], "r-o", lw=2, label="Línea de operación")
ax.annotate("fondo", (Xb2, Yb2), xytext=(6, 6), textcoords="offset points")
ax.annotate("tope", (Xt2, Yt2), xytext=(6, -12), textcoords="offset points")
ax.fill_between([Xb2, Xt2], [Y_eq(Xb2, m), Y_eq(Xt2, m)], [Yb2, Yt2], color="tab:orange", alpha=0.15)
ax.set_title("DESORCIÓN: operación bajo el equilibrio")
ax.set_xlabel("X"); ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

### 14.2 Contracorriente frente a corriente paralela

Para el mismo gas de entrada, la misma relación $L_s/G_s$ y un número infinito de etapas, la corriente paralela queda limitada por el equilibrio de salida.

In [ ]:
m = 2.0
Yin, Xin = 0.0638, 0.0
pend = 2.786                      # L_s/G_s

# Contracorriente: la línea de operación llega a Yt = 0.1·Yin (90 % de recuperación)
Yt_cc = 0.1 * Yin
Xb_cc = Xin + (Yin - Yt_cc) / pend

# Corriente paralela: Y = Yin - pend·(X - Xin), cruza el equilibrio en el punto de salida
Xo = brentq(lambda X: Yin - pend * (X - Xin) - Y_eq(X, m), 0, 0.2)
Yo = Y_eq(Xo, m)

Xg = np.linspace(0, 0.05, 200)
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(Xg, Y_eq(Xg, m), "k", lw=2, label="Equilibrio")
ax.plot([Xin, Xb_cc], [Yt_cc, Yin], "r-o", lw=2, label="Contracorriente")
ax.plot([Xin, Xo], [Yin, Yo], "b-s", lw=2, label="Corriente paralela")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("Límite de la corriente paralela")
ax.legend(loc="upper left")
plt.show()

print(f"Recuperación contracorriente (diseño)         : {100*(Yin - Yt_cc)/Yin:.1f} %")
print(f"Recuperación máxima en corriente paralela     : {100*(Yin - Yo)/Yin:.1f} %  (con infinitas etapas)")

### 14.3 Líneas de unión (fuerza impulsora local)

Se muestran las líneas de unión desde varios puntos de la línea de operación hasta la curva de equilibrio, con pendiente $-k_x/k_y$ (valores del problema de dos películas del cuaderno anterior).

In [ ]:
m = 2.0
ky, kx = 1.5e-3, 6.0e-3        # kmol/(m²·s)
s = kx / ky                    # pendiente = -s

Xt, Yt, Xb, Yb = 0.0, 0.0064, 0.0206, 0.0638
Xg = np.linspace(0, 0.05, 200)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, s_i, titulo in zip(axs, [kx / ky, 0.25 * kx / ky],
                            ["Resistencia repartida (k_x/k_y = 4)", "Resistencia en el líquido (k_x/k_y = 1)"]):
    ax.plot(Xg, Y_eq(Xg, m), "k", lw=2, label="Equilibrio")
    ax.plot([Xt, Xb], [Yt, Yb], "r-", lw=2, label="Línea de operación")
    for X in np.linspace(Xt + 0.002, Xb - 0.001, 5):
        Y = Yt + (Yb - Yt) / (Xb - Xt) * (X - Xt)
        # Intersección con el equilibrio: Yi - Y = -s·(Xi - X)
        Xi = brentq(lambda xi: Y_eq(xi, m) - Y + s_i * (xi - X), 0, 0.2)
        Yi = Y_eq(Xi, m)
        ax.plot([X, Xi], [Y, Yi], "g--", lw=1)
        ax.plot(X, Y, "ro", ms=4)
        ax.plot(Xi, Yi, "g^", ms=5)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("X")
axs[0].set_ylabel("Y")
axs[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

---
# PARTE 2 · EJEMPLOS RESUELTOS

Los ejemplos 1, 2, 3 y 7 comparten el mismo sistema (un **caso de estudio**), de manera que el diseño se construye por etapas. Cada ejemplo sigue la secuencia: **enunciado → datos → solución paso a paso → verificación con Python → análisis**.

## Ejemplo 1 · Balance de materia en un absorbedor

**Enunciado.** Se desea eliminar un contaminante A de una corriente de gas con agua como disolvente (pura a la entrada). Calcule:

a) el flujo de gas inerte $G_s$ y la razón molar de entrada $Y_b$;

b) la razón molar $Y_t$, la fracción molar $y_t$ y el flujo total $G_t$ del gas que sale;

c) los kmol/h de A que se absorben.

**Datos** (caso de estudio)

| Variable | Valor |
|---|---|
| Flujo de gas de entrada, $G_b$ | 100 kmol/h |
| Fracción molar de A en el gas, $y_b$ | 0.06 |
| Recuperación de A | 90 % |
| Disolvente entrante | agua pura, $X_t=0$ |
| Equilibrio | $y = 2\,x$ ($m=2$), a la $T$ y $P$ de operación |

### Solución paso a paso

**Paso 1. Gas inerte:**

$$
G_s = G_b\,(1-y_b) = 100\,(1-0.06) = 94.0\ \text{kmol/h}
$$

**Paso 2. Razón molar de entrada:**

$$
Y_b = \frac{y_b}{1-y_b} = \frac{0.06}{0.94} = 0.06383
$$

**Paso 3. Soluto que entra y que sale.** Entran $100\times0.06 = 6.0$ kmol/h de A. Con 90 % de recuperación salen $0.10\times 6.0 = 0.60$ kmol/h.

$$
Y_t = \frac{0.60}{G_s} = \frac{0.60}{94.0} = 0.006383
$$

**Paso 4. Gas de salida:**

$$
y_t = \frac{Y_t}{1+Y_t} = 0.006342,
\qquad
G_t = G_s\,(1+Y_t) = 94.0\,(1.006383) = 94.6\ \text{kmol/h}
$$

**Paso 5. Soluto absorbido:**

$$
G_s\,(Y_b - Y_t) = 94.0\,(0.06383-0.006383) = 5.40\ \text{kmol/h}
$$

### Verificación con Python

In [ ]:
G_b, y_b, rec, m = 100.0, 0.06, 0.90, 2.0
X_t = 0.0

G_s = G_b * (1 - y_b)                 # Paso 1
Y_b = y_b / (1 - y_b)                 # Paso 2
Y_t = (1 - rec) * Y_b                 # Paso 3
y_t = Y_t / (1 + Y_t)                 # Paso 4
G_t = G_s * (1 + Y_t)
A_abs = G_s * (Y_b - Y_t)             # Paso 5

print(f"G_s            = {G_s:.2f} kmol/h")
print(f"Y_b            = {Y_b:.5f}")
print(f"Y_t            = {Y_t:.6f}")
print(f"y_t            = {y_t:.6f}")
print(f"G_t            = {G_t:.2f} kmol/h")
print(f"A absorbido    = {A_abs:.2f} kmol/h  (entra {G_b*y_b:.1f}, sale {G_s*Y_t:.2f})")

### Análisis

- El flujo de gas inerte ($94.0$ kmol/h) es el que se mantiene constante: es la base de todos los balances que siguen.
- Para $y_t$ se debe usar $Y_t/(1+Y_t)$, no $Y_t$ directamente. Con soluciones diluidas la diferencia es pequeña ($0.006342$ frente a $0.006383$), pero con corrientes concentradas sería importante.

## Ejemplo 2 · Flujo mínimo y flujo de operación de líquido

Continuación del Ejemplo 1. Calcule:

a) la composición del líquido en equilibrio con el gas de entrada, $X_b^*$;

b) el flujo mínimo de disolvente, $L_{s,min}$;

c) el flujo de operación con $f=1.5$, la composición del líquido de salida $X_b$ y su flujo total.

### Solución paso a paso

**Paso 1. Líquido en equilibrio con el gas de entrada.** Con $X^*=\dfrac{Y}{m+(m-1)Y}$:

$$
X_b^{*} = \frac{0.06383}{2+(1)(0.06383)} = \frac{0.06383}{2.06383} = 0.03093
$$

**Paso 2. Pendiente mínima:**

$$
\left(\frac{L_s}{G_s}\right)_{min} = \frac{Y_b-Y_t}{X_b^{*}-X_t} = \frac{0.06383-0.006383}{0.03093-0} = 1.857
$$

**Paso 3. Flujo mínimo de disolvente:**

$$
L_{s,min} = 1.857\times 94.0 = 174.6\ \text{kmol/h}
$$

**Paso 4. Flujo de operación** (con $f=1.5$):

$$
\frac{L_s}{G_s} = 1.5\,(1.857) = 2.786,
\qquad
L_s = 2.786\times 94.0 = 261.9\ \text{kmol/h}
$$

**Paso 5. Líquido de salida.** Del balance de soluto:

$$
X_b = X_t + \frac{G_s\,(Y_b-Y_t)}{L_s} = 0+\frac{5.40}{261.9} = 0.02062,
\qquad
x_b = \frac{X_b}{1+X_b} = 0.02020
$$

$$
L_b = L_s\,(1+X_b) = 261.9\,(1.02062) = 267.3\ \text{kmol/h}
$$

### Verificación con Python

In [ ]:
# Datos del Ejemplo 1
G_s, Y_b, Y_t, X_t, m = 94.0, 0.06 / 0.94, 0.1 * 0.06 / 0.94, 0.0, 2.0

X_b_eq = X_eq(Y_b, m)                                   # Paso 1
pend_min = (Y_b - Y_t) / (X_b_eq - X_t)                 # Paso 2
Ls_min = pend_min * G_s                                 # Paso 3
f = 1.5
pend = f * pend_min                                     # Paso 4
L_s = pend * G_s
X_b = X_t + G_s * (Y_b - Y_t) / L_s                     # Paso 5
x_b = X_b / (1 + X_b)
L_b = L_s * (1 + X_b)

print(f"X_b* (equilibrio con el gas de entrada) = {X_b_eq:.5f}")
print(f"(L_s/G_s)_min   = {pend_min:.4f}")
print(f"L_s,min         = {Ls_min:.1f} kmol/h")
print(f"L_s/G_s (f=1.5) = {pend:.4f}")
print(f"L_s             = {L_s:.1f} kmol/h")
print(f"X_b = {X_b:.5f}   x_b = {x_b:.5f}   L_b = {L_b:.1f} kmol/h")
print(f"Comprobación: L_s·(X_b - X_t) = {L_s*(X_b - X_t):.3f} = G_s·(Y_b - Y_t) = {G_s*(Y_b - Y_t):.3f}")

### Análisis

- El líquido que sale ($X_b=0.0206$) queda **por debajo** de $X_b^*=0.0309$: la diferencia es la fuerza impulsora disponible en el fondo.
- Operar con $f=1.5$ requiere 50 % más disolvente que el mínimo teórico, a cambio de un número finito y pequeño de etapas (Ejemplo 3).

## Ejemplo 3 · Número de etapas ideales por el método escalonado

Continuación del Ejemplo 2 (con $f=1.5$). Determine:

a) el número de etapas ideales con la construcción de McCabe-Thiele;

b) cómo cambia al variar el factor $f$ entre $1.1$ y $3.0$.

### Solución paso a paso

**Paso 1. Línea de operación.** Pasa por $(X_t,Y_t)=(0,\ 0.006383)$ con pendiente $2.786$:

$$
Y = 0.006383 + 2.786\,X
$$

**Paso 2. Primera etapa (desde el fondo).** Parte del extremo $(X_b,Y_b)=(0.02062,\ 0.06383)$ y sube o baja en vertical hasta el equilibrio:

$$
Y_N = \frac{2\,(0.02062)}{1-0.02062} = 0.04211
$$

**Paso 3. Se avanza en horizontal hasta la línea de operación:**

$$
X_{N-1} = \frac{Y_N - Y_t}{L_s/G_s} = \frac{0.04211-0.006383}{2.786} = 0.01282
$$

**Paso 4. Se repite** con $X = 0.01282$, luego con $0.00703$ y con $0.00279$. Después de la cuarta etapa, la composición $X_{n-1}$ ya queda por debajo de $X_t=0$: la última etapa se completa solo parcialmente, con la fracción obtenida por interpolación.

**Resultado:** $N\approx 3.91$ etapas ideales, es decir, **4 etapas teóricas**.

**Paso 5. Sensibilidad.** Repetir el procedimiento para distintos valores de $f$:

| $f$ | $N$ ideal |
|---|---|
| 1.1 | 8.95 |
| 1.2 | 6.36 |
| 1.5 | **3.91** |
| 2.0 | 2.74 |
| 3.0 | 1.92 |

### Verificación con Python

In [ ]:
m = 2.0
G_s, Y_b, Y_t, X_t = 94.0, 0.06 / 0.94, 0.1 * 0.06 / 0.94, 0.0
eq = lambda X: Y_eq(X, m)

pend_min = (Y_b - Y_t) / (X_eq(Y_b, m) - X_t)
res = etapas_ideales(G_s, 1.5 * pend_min * G_s, Y_b, Y_t, X_t, eq)

print(f"Etapas ideales: N = {res['N']:.2f}\n")
print("Etapa (desde el fondo)   X (líquido)   Y (gas)")
for n, (x, y) in enumerate(zip(res["X"], res["Y"]), 1):
    print(f"{n:>12d}        {x:>10.5f}   {y:>10.5f}")

# --- Sensibilidad ---
factores = np.array([1.05, 1.1, 1.2, 1.3, 1.5, 1.75, 2.0, 2.5, 3.0])
N_f = [etapas_ideales(G_s, f * pend_min * G_s, Y_b, Y_t, X_t, eq)["N"] for f in factores]

fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.4))

# Diagrama escalonado
ax = axs[0]
Xg = np.linspace(0, 0.04, 200)
ax.plot(Xg, Y_eq(Xg, m), "k", lw=2, label="Equilibrio")
ax.plot([X_t, res["Xb"]], [Y_t, Y_b], "r-", lw=2, label="Línea de operación (f = 1.5)")
ax.plot([X_t, X_eq(Y_b, m)], [Y_t, Y_b], "r--", lw=1, label="Línea de operación mínima")
xs, ys = trazo_escalones(res)
ax.plot(xs, ys, "b-", lw=1.5, label=f"Escalones (N = {res['N']:.2f})")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("Método escalonado de McCabe-Thiele")
ax.legend(fontsize=8, loc="upper left")

# N frente a f
ax = axs[1]
ax.plot(factores, N_f, "o-", lw=2)
ax.set_xlabel("f = (L_s/G_s) / (L_s/G_s)_min")
ax.set_ylabel("Etapas ideales")
ax.set_title("Compromiso entre disolvente y etapas")
plt.tight_layout()
plt.show()

### Análisis

- Cerca del flujo mínimo el número de etapas **crece muy rápido** (9 etapas con $f=1.1$ frente a 4 con $f=1.5$), por eso no se opera allí.
- Pasar de $f=2$ a $f=3$ ahorra menos de una etapa, pero consume 50 % más disolvente que regenerar. Por eso el óptimo económico queda en $f=1.2$–$2.0$.

## Ejemplo 4 · Ecuación de Kremser en absorción diluida

**Enunciado.** Un gas diluido se trata en una columna en contracorriente. Calcule el número de etapas ideales con la ecuación de Kremser y compárelo con el método escalonado.

**Datos**

| Variable | Valor |
|---|---|
| Equilibrio | $y = 1.5\,x$ (lineal) |
| Gas de entrada, $y_b$ | 0.020 |
| Gas de salida, $y_t$ | 0.001 |
| Líquido de entrada, $x_t$ | 0 (disolvente puro) |
| Relación de operación | $L/G = 1.4\,(L/G)_{min}$ |

### Solución paso a paso

**Paso 1. Recuperación:**

$$
\varphi = \frac{y_b-y_t}{y_b-m\,x_t} = \frac{0.020-0.001}{0.020-0} = 0.95
$$

**Paso 2. Relación mínima** (equilibrio lineal y $x_t=0$):

$$
\left(\frac{L}{G}\right)_{min} = m\,\varphi = 1.5\times0.95 = 1.425
$$

**Paso 3. Relación de operación:**

$$
\frac{L}{G} = 1.4\times1.425 = 1.995
$$

**Paso 4. Factor de absorción:**

$$
A = \frac{L}{m\,G} = \frac{1.995}{1.5} = 1.33
$$

**Paso 5. Ecuación de Kremser:**

$$
N = \frac{\ln\left[\dfrac{1-\varphi/A}{1-\varphi}\right]}{\ln A}
= \frac{\ln\left[\dfrac{1-0.95/1.33}{1-0.95}\right]}{\ln 1.33}
= \frac{\ln(5.714)}{0.2852} = 6.11
$$

### Verificación con Python

In [ ]:
m4, y_b, y_t, x_t = 1.5, 0.020, 0.001, 0.0

phi = (y_b - y_t) / (y_b - m4 * x_t)         # Paso 1
LG_min = m4 * phi                            # Paso 2
LG = 1.4 * LG_min                            # Paso 3
A = LG / m4                                  # Paso 4
N_kremser = kremser_N(phi, A)                # Paso 5

# Comparación con el método escalonado (diluido: Gs = 1, Ls = L/G, Y = y, X = x)
N_esc = etapas_ideales(1.0, LG, y_b, y_t, x_t, lambda X: m4 * X)["N"]

print(f"φ = {phi:.3f}   (L/G)_min = {LG_min:.3f}   L/G = {LG:.3f}   A = {A:.3f}")
print(f"N (Kremser)    = {N_kremser:.3f}")
print(f"N (escalonado) = {N_esc:.3f}")
print(f"Comprobación de φ con N de Kremser: {kremser_phi(N_kremser, A):.3f}")

### Análisis

- Ambos métodos coinciden (6.11 frente a 6.13). La pequeña diferencia viene de **interpolar linealmente** la última etapa en el método escalonado; la fórmula de Kremser es exacta.
- Kremser solo es válida con **equilibrio lineal** y **línea de operación recta**: aplica a soluciones diluidas.

## Ejemplo 5 · Desorción (stripping) de un contaminante del agua

**Enunciado.** Agua con un contaminante volátil se trata en una columna con aire limpio en contracorriente. Calcule:

a) la relación máxima $L/G$ (flujo mínimo de aire);

b) el factor de desorción $S$ con aire a 1.5 veces el mínimo;

c) el número de etapas ideales;

d) el flujo de aire y la composición del aire de salida si se tratan 500 kmol/h de agua.

**Datos**

| Variable | Valor |
|---|---|
| Equilibrio | $y = 20\,x$ (lineal, diluido) |
| Agua de entrada (tope), $x_t$ | $1.0\times10^{-4}$ |
| Agua de salida (fondo), $x_b$ | $1.0\times10^{-5}$ (90 % de remoción) |
| Aire de entrada (fondo), $y_b$ | 0 |
| Flujo de agua, $L$ | 500 kmol/h |

### Solución paso a paso

**Paso 1. Pendiente máxima.** En el tope el líquido de entrada $x_t$ queda en equilibrio con el gas que sale ($y_t^* = m\,x_t$):

$$
\left(\frac{L}{G}\right)_{max} = \frac{y_t^{*}-y_b}{x_t-x_b} = \frac{20\,(1.0\times10^{-4})-0}{1.0\times10^{-4}-1.0\times10^{-5}} = 22.2
$$

**Paso 2. Relación de operación** (aire = 1.5 veces el mínimo, por tanto $L/G$ se divide entre 1.5):

$$
\frac{L}{G} = \frac{22.2}{1.5} = 14.81
\;\;\Rightarrow\;\;
S = \frac{m\,G}{L} = \frac{20}{14.81} = 1.35
$$

**Paso 3. Fracción desorbida:**

$$
\varphi = \frac{x_t-x_b}{x_t - y_b/m} = \frac{1.0\times10^{-4}-1.0\times10^{-5}}{1.0\times10^{-4}} = 0.90
$$

**Paso 4. Ecuación de Kremser para desorción:**

$$
N = \frac{\ln\left[\dfrac{1-\varphi/S}{1-\varphi}\right]}{\ln S}
= \frac{\ln\left[\dfrac{1-0.90/1.35}{0.10}\right]}{\ln 1.35}
= \frac{\ln(3.333)}{0.3001} = 4.01
$$

**Paso 5. Flujo de aire y composición de salida:**

$$
G = \frac{L}{L/G} = \frac{500}{14.81} = 33.75\ \text{kmol/h}
$$

$$
y_t = \frac{L}{G}\,(x_t-x_b) = 14.81\,(9.0\times10^{-5}) = 1.33\times10^{-3}
$$

### Verificación con Python

In [ ]:
m5, x_t, x_b, y_b = 20.0, 1.0e-4, 1.0e-5, 0.0
L = 500.0

LG_max = (m5 * x_t - y_b) / (x_t - x_b)                 # Paso 1
LG = LG_max / 1.5                                       # Paso 2
S = m5 / LG
phi = (x_t - x_b) / (x_t - y_b / m5)                    # Paso 3
N_kremser = kremser_N(phi, S)                           # Paso 4
G = L / LG                                              # Paso 5
y_t = LG * (x_t - x_b) + y_b

# Comparación con el método escalonado (Gs = 1, Ls = L/G; Y_t se obtiene del balance)
N_esc = etapas_ideales(1.0, LG, y_b, y_t, x_t, lambda X: m5 * X)["N"]

print(f"(L/G)_max         = {LG_max:.2f}   →  G_min = {L/LG_max:.2f} kmol/h")
print(f"L/G de operación  = {LG:.2f}   S = {S:.3f}   φ = {phi:.2f}")
print(f"N (Kremser)       = {N_kremser:.3f}")
print(f"N (escalonado)    = {N_esc:.3f}")
print(f"Flujo de aire     = {G:.2f} kmol/h")
print(f"y_t (aire salida) = {y_t:.3e}")

### Análisis

- El aire de salida tiene una concentración muy baja ($1.3\times10^{-3}$) porque el contaminante está muy diluido; el flujo de aire se calcula a partir de $L/G$.
- Con $S\approx1.35$ se necesitan unas **4 etapas ideales**. Si se usara menos aire ($S\to1$), $N$ aumentaría rápidamente, de la misma manera que al acercarse al flujo mínimo en absorción.

## Ejemplo 6 · Eficiencia de Murphree y número de platos reales

**Enunciado.** Para la columna del Ejemplo 4 ($N_{ideal}=6.11$, $m=1.5$, $A=1.33$), pruebas en planta indican una eficiencia de Murphree en fase gas **constante** $E_{MV}=0.65$. Calcule:

a) la eficiencia global $E_o$ con la relación de Lewis;

b) el número de platos reales;

c) verifique con un cálculo plato por plato que se alcanza la recuperación de 95 %.

### Solución paso a paso

**Paso 1. Parámetro $\lambda$:**

$$
\lambda = \frac{m\,G}{L} = \frac{1}{A} = \frac{1}{1.33} = 0.752
$$

**Paso 2. Eficiencia global (Lewis):**

$$
E_o = \frac{\ln\left[1+E_{MV}\,(\lambda-1)\right]}{\ln\lambda}
= \frac{\ln\left[1+0.65\,(0.752-1)\right]}{\ln 0.752}
= \frac{\ln(0.8387)}{\ln(0.752)} = 0.617
$$

**Paso 3. Platos reales:**

$$
N_{reales} = \frac{N_{ideales}}{E_o} = \frac{6.11}{0.617} = 9.9 \;\;\Rightarrow\;\; 10 \text{ platos}
$$

**Paso 4. Verificación plato por plato.** Con $N$ platos y $E_{MV}$ constante se plantean dos ecuaciones por plato ($n=1\ldots N$, numeradas desde el tope):

$$
Y_n = Y_{n+1} + E_{MV}\,(m\,X_n - Y_{n+1})
\qquad\text{(Murphree)}
$$

$$
L_s\,(X_n - X_{n-1}) = G_s\,(Y_{n+1}-Y_n)
\qquad\text{(balance de soluto)}
$$

con $X_0 = x_t$ y $Y_{N+1}=y_b$. Es un sistema lineal de $2N$ ecuaciones que se resuelve con `numpy`.

### Verificación con Python

In [ ]:
m6, y_b, x_t = 1.5, 0.020, 0.0
LG = 1.995
A = LG / m6
N_ideal = kremser_N(0.95, A)
E_MV = 0.65

# Pasos 1 y 2
lam = 1 / A
E_o = np.log(1 + E_MV * (lam - 1)) / np.log(lam)
N_real = N_ideal / E_o
print(f"λ = {lam:.3f}   E_o = {E_o:.3f}   N_reales = {N_real:.2f}")

# Paso 4: cálculo plato por plato con eficiencia de Murphree constante
def columna_platos(N, E, m, Gs, Ls, Yb, Xt):
    # Incógnitas: X_1..X_N (líquido que sale de cada plato) y Y_1..Y_N (gas que sale)
    # Platos numerados desde el tope; Y_{N+1} = Yb, X_0 = Xt
    A_m = np.zeros((2 * N, 2 * N)); b = np.zeros(2 * N)
    for i in range(N):
        # Murphree: Y_i - (1-E)·Y_{i+1} - E·m·X_i = 0
        A_m[i, N + i] = 1.0
        A_m[i, i] = -E * m
        if i < N - 1:
            A_m[i, N + i + 1] = -(1 - E)
        else:
            b[i] = (1 - E) * Yb
        # Balance: Ls·(X_i - X_{i-1}) - Gs·(Y_{i+1} - Y_i) = 0
        r = N + i
        A_m[r, i] = Ls
        A_m[r, N + i] = Gs
        if i > 0:
            A_m[r, i - 1] = -Ls
        else:
            b[r] = Ls * Xt
        if i < N - 1:
            A_m[r, N + i + 1] = -Gs
        else:
            b[r] += Gs * Yb
    sol = np.linalg.solve(A_m, b)
    return sol[:N], sol[N:]

print()
print("Platos   y de salida   Recuperación")
for N in (8, 9, 10, 11, 12):
    X_p, Y_p = columna_platos(N, E_MV, m6, 1.0, LG, y_b, x_t)
    print(f"{N:>6d}   {Y_p[0]:>10.5f}   {100*(y_b - Y_p[0])/y_b:>9.2f} %")

# Comprobación: con E = 1 se recupera el resultado de Kremser para N entero
X_i, Y_i = columna_platos(6, 1.0, m6, 1.0, LG, y_b, x_t)
print(f"\nCon E = 1 y N = 6: y_t = {Y_i[0]:.5f}   (Kremser: {y_b*(1 - kremser_phi(6, A)):.5f})")

### Análisis

- Con 10 platos reales la recuperación es de **95.1 %** (cumple el objetivo), y con 9 platos solo de 93.97 %: el resultado confirma el valor $N_{real}\approx9.9$ de la relación de Lewis.
- Una eficiencia de $0.617$ implica construir $\approx 60$ % más platos que etapas ideales. Mejorar el contacto (tipo de plato, gasto, espumado) aumenta $E_{MV}$ y reduce el costo.
- La comprobación con $E=1$ reproduce la ecuación de Kremser, lo que valida el código.

## Ejemplo 7 · Balance de energía: absorción adiabática

Continuación del caso de estudio (Ejemplos 1 a 3). La absorción de A **libera calor**. Calcule:

a) el aumento de temperatura del líquido en una columna adiabática;

b) la pendiente de equilibrio $m$ en el fondo;

c) si el diseño isotérmico del Ejemplo 2 ($L_s=261.9$ kmol/h) sigue siendo factible;

d) el flujo mínimo de disolvente que sí es factible.

**Datos adicionales** (valores supuestos)

| Variable | Valor |
|---|---|
| Calor de disolución, $\Delta H_{sol}$ | $-35$ kJ/mol $= -35\,000$ kJ/kmol |
| Disolvente | agua: $M = 18.015$ kg/kmol, $c_p = 4.18$ kJ/(kg·K) |
| Temperatura de entrada, $T_0$ | 298.15 K |
| Equilibrio a $T_0$ | $m_0 = 2.0$ |

Se supone que **todo el calor** se transfiere al líquido (se desprecia el calor sensible del gas y la evaporación) y que la temperatura varía linealmente con $X$.

### Solución paso a paso

**Paso 1. Calor liberado:**

$$
Q = G_s\,(Y_b-Y_t)\,(-\Delta H_{sol}) = 5.40\ \text{kmol/h}\times35\,000\ \text{kJ/kmol} = 1.89\times10^{5}\ \text{kJ/h}
$$

**Paso 2. Capacidad calorífica del líquido:**

$$
L_s\,M\,c_p = 261.9\times 18.015\times 4.18 = 1.97\times10^{4}\ \text{kJ/(h·K)}
$$

**Paso 3. Aumento de temperatura:**

$$
\Delta T = \frac{1.89\times10^{5}}{1.97\times10^{4}} = 9.6\ \text{K}
\;\;\Rightarrow\;\; T_b = 307.7\ \text{K}
$$

**Paso 4. Pendiente de equilibrio en el fondo** (van't Hoff, con $\Delta H_{sol}/R = -35\,000/8.314 = -4210$ K):

$$
m(T_b) = 2.0\,\exp\!\left[-4210\left(\frac{1}{307.7}-\frac{1}{298.15}\right)\right] = 2.0\times 1.55 = 3.10
$$

**Paso 5. Factibilidad.** Con $T$ lineal en $X$, la curva de equilibrio "adiabática" se desplaza hacia arriba conforme baja el líquido. Al compararla con la línea de operación del Ejemplo 2, **la línea de operación cruza la curva de equilibrio** (en el código se ve una diferencia negativa): con $L_s = 261.9$ kmol/h la separación **no es posible**.

**Paso 6. Flujo mínimo factible.** Se busca, por tanteo numérico, la menor pendiente para la cual la línea de operación queda por encima de la curva adiabática (con $\Delta T$ recalculado para cada $L_s$):

$$
\left(\frac{L_s}{G_s}\right)_{min,ad} = 2.88 \quad\text{frente a}\quad 1.857 \;(\text{isotérmico})
\;\Rightarrow\; \text{ratio} = 1.55
$$

### Verificación con Python

In [ ]:
# --- Datos del caso de estudio ---
G_s, Y_b, Y_t, X_t = 94.0, 0.06 / 0.94, 0.1 * 0.06 / 0.94, 0.0
m0, T0 = 2.0, 298.15
dH = -35000.0                      # kJ/kmol  (= J/mol·1000/1000)
dH_R = dH / R                      # K   (dH en J/mol = -35000 → dH/R en K)
M_agua, cp = 18.015, 4.18

pend_min_iso = (Y_b - Y_t) / (X_eq(Y_b, m0) - X_t)
L_s = 1.5 * pend_min_iso * G_s     # diseño isotérmico (Ejemplo 2)
X_b = X_t + G_s * (Y_b - Y_t) / L_s

def dT_adiabatico(Ls_):
    Q = G_s * (Y_b - Y_t) * (-dH)                  # kJ/h
    return Q / (Ls_ * M_agua * cp)

def m_de_T(T):
    return m0 * np.exp(dH_R * (1 / T - 1 / T0))

# --- Pasos 1 a 4 ---
Q = G_s * (Y_b - Y_t) * (-dH)
dT = dT_adiabatico(L_s)
print(f"Q   = {Q:.3e} kJ/h")
print(f"ΔT  = {dT:.2f} K   →  T_fondo = {T0 + dT:.1f} K")
print(f"m en el fondo = {m_de_T(T0 + dT):.2f}   (a la entrada: {m0:.2f})")

# --- Curva de equilibrio adiabática para un L_s dado ---
def eq_adiabatica(Ls_):
    Xb_ = X_t + G_s * (Y_b - Y_t) / Ls_
    dT_ = dT_adiabatico(Ls_)
    def f(X):
        T = T0 + dT_ * (X - X_t) / (Xb_ - X_t)
        return Y_eq(X, m_de_T(T))
    return f, Xb_

def holgura_minima(pend):
    Ls_ = pend * G_s
    f, Xb_ = eq_adiabatica(Ls_)
    Xg = np.linspace(X_t, Xb_, 400)
    op = Y_t + pend * (Xg - X_t)
    return np.min(op - np.array([f(x) for x in Xg]))

# --- Paso 5: factibilidad del diseño isotérmico ---
pend_iso = L_s / G_s
print(f"\nHolgura mínima (op - eq) con L_s = {L_s:.1f}: {holgura_minima(pend_iso):.4f}  "
      f"→ {'FACTIBLE' if holgura_minima(pend_iso) > 0 else 'NO FACTIBLE (la línea cruza el equilibrio)'}")

# --- Paso 6: pendiente mínima factible ---
pend_min_ad = brentq(holgura_minima, pend_iso, 8.0)
print(f"(L_s/G_s)_min adiabático = {pend_min_ad:.3f}   frente a {pend_min_iso:.3f} isotérmico "
      f"(×{pend_min_ad/pend_min_iso:.2f})")

# --- Gráfica ---
f_ad, Xb_ad = eq_adiabatica(L_s)
Xg = np.linspace(0, 0.04, 300)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(Xg, Y_eq(Xg, m0), "k", lw=2, label="Equilibrio isotérmico (T₀)")
Xg2 = np.linspace(0, Xb_ad, 200)
ax.plot(Xg2, [f_ad(x) for x in Xg2], "tab:orange", lw=2, label="Equilibrio adiabático (T crece con X)")
ax.plot([X_t, X_b], [Y_t, Y_b], "r-o", lw=2, label=f"Operación L_s = {L_s:.0f} kmol/h")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("El calor de disolución desplaza el equilibrio hacia arriba")
ax.legend(fontsize=8, loc="upper left")
plt.show()

### Análisis

- Un aumento de $\approx 10$ K eleva la pendiente de equilibrio de $2.0$ a $3.1$ en el fondo: el diseño isotérmico **deja de funcionar**.
- El flujo mínimo de líquido sube cerca de **55 %**. Las soluciones prácticas son usar más disolvente, enfriar el líquido a la mitad de la columna o entrar con el líquido más frío.
- Es un modelo simplificado (calor solo al líquido, $T$ lineal con $X$). Un cálculo riguroso incluye entalpías de las dos fases y el calor sensible del gas, que se verá con el método de Ponchon-Savarit.

---
# PARTE 3 · EJERCICIOS PROPUESTOS

Resuelve cada ejercicio en la celda de código que sigue. Las respuestas aparecen al final para que compruebes tu resultado. Usa las funciones de apoyo del inicio del cuaderno.

**Ejercicio 1.** Una corriente de 200 kmol/h con $y_b=0.10$ de un soluto A se lava con disolvente puro. Se requiere una recuperación de 95 %. El equilibrio es $y=1.5\,x$ (usar $Y^* = mX/[1+(1-m)X]$).
a) Calcule $G_s$, $Y_b$ y $Y_t$.
b) Calcule $L_{s,min}$.
c) Con $L_s = 1.3\,L_{s,min}$, determine el número de etapas ideales.

**Ejercicio 2.** Una columna de absorción diluida opera con un factor de absorción $A=1.25$ y debe recuperar 90 % del soluto ($\varphi=0.90$). Calcule el número de etapas ideales con Kremser.

**Ejercicio 3.** Una columna de desorción opera con $S=1.6$ y debe remover 99 % del soluto ($\varphi=0.99$). Calcule $N$.

**Ejercicio 4.** Con $E_{MV}=0.50$ y $\lambda = mG/L = 1.2$, calcule la eficiencia global $E_o$ con la relación de Lewis.

**Ejercicio 5.** Con equilibrio $Y^*=2X$, $X_{in}=0$ y $L_s/G_s=3$, ¿cuál es la fracción máxima que se puede absorber en **corriente paralela**, aunque la columna tenga etapas infinitas?

In [ ]:
# ---- Espacio para resolver tus ejercicios ----

# Ejercicio 1


# Ejercicio 2


# Ejercicio 3


# Ejercicio 4


# Ejercicio 5

### Respuestas para comprobar

| Ejercicio | Resultado |
|---|---|
| 1 | a) $G_s=180$ kmol/h, $Y_b=0.1111$, $Y_t=0.005556$; b) $L_{s,min}\approx 266$ kmol/h; c) $L_s\approx 346$ kmol/h y $N\approx 6.9$ etapas |
| 2 | $N \approx 4.61$ |
| 3 | $N \approx 7.75$ |
| 4 | $E_o \approx 0.523$ |
| 5 | $\varphi_{max} = \dfrac{3}{2+3} = 0.60$ (60 %) |

---
# PARTE 4 · RESUMEN Y REFERENCIAS

## Resumen de fórmulas

| Concepto | Expresión |
|---|---|
| Razones molares | $Y=\dfrac{y}{1-y}$, $\;X=\dfrac{x}{1-x}$ |
| Equilibrio (Henry) | $Y^*=\dfrac{mX}{1+(1-m)X}$, $\;X^*=\dfrac{Y}{m+(m-1)Y}$ |
| Balance de soluto | $G_s(Y_b-Y_t)=L_s(X_b-X_t)$ |
| Línea de operación | $Y=Y_t+\dfrac{L_s}{G_s}(X-X_t)$ |
| Absorción: pendiente mínima | $\left(\dfrac{L_s}{G_s}\right)_{min}=\dfrac{Y_b-Y_t}{X_b^*-X_t}$ |
| Desorción: pendiente máxima | $\left(\dfrac{L_s}{G_s}\right)_{max}=\dfrac{Y_t^*-Y_b}{X_t-X_b}$ |
| Factores | $A=\dfrac{L}{mG}$ (absorción), $\;S=\dfrac{mG}{L}$ (desorción) |
| Kremser | $\varphi=\dfrac{F^{N+1}-F}{F^{N+1}-1}$, $\;N=\dfrac{\ln\left[(1-\varphi/F)/(1-\varphi)\right]}{\ln F}$ |
| Eficiencia global | $E_o=N_{ideal}/N_{real}$ |
| Eficiencia de Murphree | $E_{MV}=\dfrac{Y_{n+1}-Y_n}{Y_{n+1}-Y_n^*}$ |
| Lewis | $E_o=\dfrac{\ln[1+E_{MV}(\lambda-1)]}{\ln\lambda}$, $\;\lambda=\dfrac{mG}{L}$ |
| Corriente paralela (límite) | $\varphi_{max}=\dfrac{L_s/G_s}{m+L_s/G_s}$ |
| Aumento de temperatura | $\Delta T\approx\dfrac{G_s(Y_b-Y_t)(-\Delta H_{sol})}{L_s\,M\,c_p}$ |
| Efecto de $T$ en $m$ | $m(T)=m_0\exp\!\left[\dfrac{\Delta H_{sol}}{R}\left(\dfrac1T-\dfrac1{T_0}\right)\right]$ |

## Ideas clave

1. Trabajar con **razones molares** mantiene lineal el balance de materia aun con corrientes concentradas.
2. En **absorción** la línea de operación va **sobre** el equilibrio; en **desorción**, **bajo** él.
3. Existe un **flujo mínimo de líquido** (o máximo en desorción) que exigiría etapas infinitas; se opera con 1.2 a 2 veces ese límite.
4. **Kremser** resuelve el caso lineal de forma exacta y sirve para verificar el método escalonado.
5. Los platos reales superan a las etapas ideales según la **eficiencia**; la relación de Lewis une la de Murphree con la global.
6. El **calor de disolución** puede volver inviable un diseño isotérmico: hay que revisar el balance de energía.

## Siguientes temas

- Cálculo detallado de columnas empacadas (HTU, NTU, caída de presión e inundación).
- Método de Ponchon-Savarit para sistemas con efectos térmicos importantes.

## Referencias

- Treybal, R. E. *Operaciones de transferencia de masa*. McGraw-Hill.
- McCabe, W., Smith, J. y Harriott, P. *Operaciones unitarias en ingeniería química*. McGraw-Hill.
- Wankat, P. C. *Ingeniería de procesos de separación*. Pearson.
- Seader, J. D., Henley, E. J. y Roper, D. K. *Separation Process Principles*. Wiley.
- Kremser, A. (1930). Theoretical analysis of absorption process. *National Petroleum News*, 22(21), 42-49.
- Lewis, W. K. (1936). Rectification of binary mixtures. *Industrial & Engineering Chemistry*, 28, 399-402.
- O'Connell, H. E. (1946). Plate efficiency of fractionating columns and absorbers. *Transactions of the AIChE*, 42, 741-755.